# Image processing notebook: From overlap corrected to transmission 

### 00 - LP30 Pristine

##  Initial settings

### Import libraries
Import all the required libraries

In [1]:
import sys
sys.path.append(r'..\01_Functions')
from step_functions import *
from dict_functions import *
from proc_functions import *
from img_functions import *
%matplotlib inline

### Select directories
Select the source directory. This directory is where the images **after** the overlap correction were saved.
Select the destination directory. Here is where the transmission images are going to be saved.

In [2]:
# %load select_directory('src_dir')
src_dir = r"J:\700 Campaigns - internal\2022\PSI22_04NI\00_Processed\00_Overlap_corrected\exp1000\00_LP30_pristine"

In [3]:
# %load select_directory('dst_dir')
dst_dir = r"J:\700 Campaigns - internal\2022\PSI22_04NI\00_Processed\01_Transmission_results\exp1000\playground"

### Select folders to process

In [4]:
stack_dict = prep_stack_dict(src_dir)
for key in stack_dict.keys():
    print(key)

01_ob
02_pos00
03_ob
04_pos01
05_ob
06_pos02
07_ob
08_pos03
09_ob
10_pos04
11_ob


In [7]:
ref_folder = ['02_pos00']

## TEST processing for reference images

### Build the reference images dictionary 
keep_acq_numb: amount of acquisition folders to process. this is a test so we want to so it fast. thus, we limit the amount of data <br>
In case you want to get an image _(array, header)_ in the dictionary, follow the format:<br>

`test = extract_img_dict(exp_test_dict, proc_folder[0], img_number = 50)`<br>
"variable" = function("dictionary name", "key/folder", "acq_number = 0", "img_number = 50")

**_To visualize the image_** (a tuple with an array in position 0 and a header in position 1, you require the next instruction:<br>

`show_img(test[0])
test[1]` # to show the header of that image

In [8]:
ref_test_dict, ref_test_param = testing_mode_step (src_dir, proc_folder = ref_folder, keep_acq_numb = 1)

Reading Images: 100%|████████████████████████████| 3/3 [00:03<00:00,  1.25s/it]


### Pre-processing sequence with parameters
The pre-processing sequence refers to any sequence with their respective parameters that emphasize on modifying or enhancing the image (**usually filters**). In this step you can also perform an averaging of all the acquisitions `stack_averaging`, a step binning of the aquisitions `binning_acquisitions`, and/or a step binning of the frames `binning_frames`.

**The image processing (SBKG, registration, scrubbing, and/or TFC should be done in the processing step.**

**NOTE: The order of the functions in the sequence matters**, but not the order of the parameters, you can also include alien parameters, this will not affect the process.

In [9]:
pre_proc_seq = [outlier_removal, stack_averaging]
add_to_dict(ref_test_param,['threshold'], [0])

In [ ]:
ref_test_dict = pre_processing_step (ref_test_dict, pre_proc_seq, param_dict = ref_test_param)

### Processing sequence and variables to obtain the referenceimages used in the TFC

In [ ]:
proc_seq = [scrubbing_correction_dict, SBKG_correction_dict]

In [ ]:
BB_mask = get_img(src_dir + '/bb_mask_ref_full.fits')
add_to_dict(ref_test_param,['BB_mask'], [BB_mask])

In [ ]:
ref_test_dict = processing_step (ref_test_dict, proc_seq, param_dict = ref_test_param)

### Get NCA
There will be displacements in the experiment images, for this reason, it is better to get the `nca` from the reference image.

In [ ]:
# %load select_rois(ref_img_TFC[0], list_rois = ['nca'])
nca = [81, 377, 37, 12]

In [ ]:
ref_img = extract_img_dict(ref_test_dict, ref_folder[0], img_number = 50)
show_img_rois(ref_img[0], dr = [(nca, 'blue')])

### Save the acquisition 
In imageJ you can create the image for the registration with this saved acquisition 

In [ ]:
dst_dir_test = dst_dir + '/imgs_4_registration_ref'
saving_step (ref_test_dict, dst_dir_test, img_name = 'ref_4_reg')

## TEST processing for experiment images

In [6]:
exp_test_dict, exp_test_param = reading_step (src_dir, proc_folder = proc_folder)

Reading Images: 100%|████████████████████████████| 3/3 [00:03<00:00,  1.15s/it]


The pre processing sequence will not change. It will be the same as the one we used for the reference. Except if there is some averaging requirements

In [7]:
pre_proc_seq = [outlier_removal, stack_averaging]
add_to_dict(exp_test_param,['threshold'], [0])

In [8]:
exp_test_dict = pre_processing_step (exp_test_dict, pre_proc_seq, param_dict = exp_test_param)

Processing Filetring : 100%|█████████████████████| 1/1 [00:08<00:00,  8.37s/it]


### Processing sequence and variables (scrubbing and SBKG)

In [9]:
proc_seq = [slice_values_dict, scrubbing_correction_dict, SBKG_correction_dict]

In [10]:
BB_mask = get_img(src_dir + '/pos_00.fits')
add_to_dict(exp_test_param,['BB_mask', 'start_slice', 'end_slice'], [BB_mask, 0, 80])

In [11]:
exp_test_dict = processing_step (exp_test_dict, proc_seq, param_dict = exp_test_param)

Processing SBKG Correction: 100%|████████████████| 1/1 [00:05<00:00,  5.90s/it]


In [12]:
# %load select_rois(img, list_rois = ['nca'])
nca = [0,0,0,0]

### TFC and final transission

In [13]:
proc_seq = [TFC_correction_dict]

In [14]:
add_to_dict(exp_test_param,['nca', 'use_ref'], [nca, False])

In [15]:
exp_test_dict = processing_step (exp_test_dict, proc_seq, param_dict = exp_test_param)

Using proc_dict as both img and ref to calculate TFC


Processing TFC Correction: 100%|█████████████████| 1/1 [00:00<00:00,  5.77it/s]


In [16]:
dst_dir_test = dst_dir + '/test_pos00_new'
saving_step (exp_test_dict, dst_dir_test, img_name = 'testing')

Saving images as a single acquisition


Writing Images: 100%|████████████████████████████| 1/1 [00:15<00:00, 15.45s/it]


## Reference Full Processing

In [ ]:
ref_dict, ref_param = reading_step (src_dir, proc_folder = ref_folder)

In [ ]:
pre_proc_seq = [outlier_removal, stack_averaging]
add_to_dict(ref_param,['threshold'], [0])

In [ ]:
ref_dict = pre_processing_step (ref_dict, pre_proc_seq, param_dict = ref_param)

In [ ]:
proc_seq = [scrubbing_correction_dict, SBKG_correction_dict]

In [ ]:
BB_mask = get_img(src_dir + '/bb_mask_ref_full.fits')
add_to_dict(ref_param,['BB_mask'], [BB_mask])

In [ ]:
ref_dict = processing_step (ref_dict, proc_seq, param_dict = ref_param)

In [ ]:
proc_seq = [TFC_correction_dict]
add_to_dict(ref_param,['nca', 'use_ref', 'ref_dict'], [nca, False, ref_dict])

In [ ]:
#dst_dir_test = dst_dir + '/00_Ref_dict_processed'
#saving_step (ref_dict, dst_dir_test, img_name = 'so_ref_image')

## Store reference dictionary and parameters

In [ ]:
%store ref_dict

### Update the reference dictionary

In [ ]:
exp_param = exp_test_param
add_to_dict(exp_param,['ref_dict'], [ref_dict])

In [ ]:
%store exp_param